# Titanic exp014 - CV design review for exp007 vs exp013

この notebook は `exp007` と `exp013` の差が採用判断に値するかを、CV の安定性という観点から見直すためのものです。

見ること:
- fold ごとのばらつき
- 複数 seed の `StratifiedKFold` で結論が安定するか
- 単純 holdout でも同じ傾向か


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from catboost import CatBoostClassifier
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    COMP_DIR = cwd.parent
elif cwd.name == "titanic":
    COMP_DIR = cwd
else:
    COMP_DIR = Path("/home/sora/dev/kaggle/competitions/titanic")

DATA_DIR = COMP_DIR / "data"
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"


In [ ]:
TITLE_NORMALIZATION = {
    "Mlle": "Miss",
    "Ms": "Miss",
    "Mme": "Mrs",
    "Lady": "Rare",
    "Countess": "Rare",
    "Dona": "Rare",
    "Dr": "Rare",
    "Rev": "Rare",
    "Col": "Rare",
    "Major": "Rare",
    "Capt": "Rare",
    "Sir": "Rare",
    "Don": "Rare",
    "Jonkheer": "Rare",
    "Master": "Master",
    "Miss": "Miss",
    "Mr": "Mr",
    "Mrs": "Mrs",
}

BASELINE_FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Name"]
NUMERIC_COLUMNS = ["Age", "SibSp", "Parch", "Fare"]
LOGISTIC_CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass"]
CATBOOST_CATEGORICAL_COLUMNS = ["Pclass", "Sex", "Embarked", "Title", "FamilyGroup", "SexPclass"]
CATBOOST_FEATURE_COLUMNS = ["Pclass", "Sex", "Age", "SibSp", "Parch", "Fare", "Embarked", "Title", "FamilyGroup", "SexPclass"]
SEEDS = [7, 13, 42, 52, 100]


def add_exp007_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    title = out["Name"].fillna("").astype(str).str.extract(r",\s*([^\.]+)\.", expand=False)
    out["Title"] = title.fillna("Rare").map(lambda x: TITLE_NORMALIZATION.get(x, "Rare")).astype(str)
    family_size = out["SibSp"].fillna(0) + out["Parch"].fillna(0) + 1
    out["FamilyGroup"] = pd.cut(
        family_size,
        bins=[0, 1, 4, 100],
        labels=["alone", "small", "large"],
        right=True,
    ).astype(str)
    out["Sex"] = out["Sex"].fillna("missing").astype(str)
    out["Embarked"] = out["Embarked"].fillna("missing").astype(str)
    out["Pclass"] = out["Pclass"].fillna(-1).astype(int).astype(str)
    out["SexPclass"] = out["Sex"] + "_" + out["Pclass"]
    return out


def make_one_hot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_logistic_pipeline(random_state: int) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", Pipeline([
                ("imputer", SimpleImputer(strategy="median")),
                ("scaler", StandardScaler()),
            ]), NUMERIC_COLUMNS),
            ("cat", Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", make_one_hot_encoder()),
            ]), LOGISTIC_CATEGORICAL_COLUMNS),
        ],
        remainder="drop",
        sparse_threshold=0.0,
    )
    model = LogisticRegression(max_iter=3000, C=1.0, solver="lbfgs", random_state=random_state)
    return Pipeline([
        ("preprocess", preprocessor),
        ("model", model),
    ])


def build_catboost_model(random_state: int) -> CatBoostClassifier:
    return CatBoostClassifier(
        iterations=500,
        depth=6,
        learning_rate=0.05,
        loss_function="Logloss",
        eval_metric="Accuracy",
        random_seed=random_state,
        verbose=False,
    )


In [ ]:
train_raw = pd.read_csv(TRAIN_PATH)
train_df = add_exp007_features(train_raw)

display(train_df[["Sex", "Pclass", "SexPclass", "Title", "FamilyGroup"]].head())

y = train_raw["Survived"].astype(int)
X_logistic = train_df[BASELINE_FEATURE_COLUMNS + ["Title", "FamilyGroup", "SexPclass"]].copy()
X_catboost = train_df[CATBOOST_FEATURE_COLUMNS].copy()


In [ ]:
def evaluate_seed(seed: int) -> dict:
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    logistic_scores = []
    catboost_scores = []

    for train_idx, valid_idx in cv.split(X_logistic, y):
        X_train_log = X_logistic.iloc[train_idx].copy()
        X_valid_log = X_logistic.iloc[valid_idx].copy()
        y_train = y.iloc[train_idx]
        y_valid = y.iloc[valid_idx]

        logistic = build_logistic_pipeline(seed)
        logistic.fit(X_train_log, y_train)
        logistic_pred = logistic.predict(X_valid_log)
        logistic_scores.append(accuracy_score(y_valid, logistic_pred))

        X_train_cat = X_catboost.iloc[train_idx].copy()
        X_valid_cat = X_catboost.iloc[valid_idx].copy()
        catboost = build_catboost_model(seed)
        catboost.fit(X_train_cat, y_train, cat_features=CATBOOST_CATEGORICAL_COLUMNS)
        cat_pred = catboost.predict(X_valid_cat)
        catboost_scores.append(accuracy_score(y_valid, cat_pred))

    return {
        "seed": seed,
        "logistic_mean": float(np.mean(logistic_scores)),
        "logistic_std": float(np.std(logistic_scores)),
        "catboost_mean": float(np.mean(catboost_scores)),
        "catboost_std": float(np.std(catboost_scores)),
        "delta": float(np.mean(catboost_scores) - np.mean(logistic_scores)),
        "logistic_scores": logistic_scores,
        "catboost_scores": catboost_scores,
    }

seed_results = [evaluate_seed(seed) for seed in SEEDS]
seed_summary = pd.DataFrame([
    {
        "seed": row["seed"],
        "logistic_mean": row["logistic_mean"],
        "logistic_std": row["logistic_std"],
        "catboost_mean": row["catboost_mean"],
        "catboost_std": row["catboost_std"],
        "delta": row["delta"],
    }
    for row in seed_results
]).sort_values("seed")

display(seed_summary)
print('delta mean:', round(float(seed_summary['delta'].mean()), 5))
print('delta min/max:', round(float(seed_summary['delta'].min()), 5), round(float(seed_summary['delta'].max()), 5))


In [ ]:
holdout_rows = []
for seed in SEEDS:
    X_train_log, X_valid_log, y_train, y_valid = train_test_split(
        X_logistic, y, test_size=0.2, stratify=y, random_state=seed
    )
    logistic = build_logistic_pipeline(seed)
    logistic.fit(X_train_log, y_train)
    logistic_score = accuracy_score(y_valid, logistic.predict(X_valid_log))

    X_train_cat, X_valid_cat, y_train_cat, y_valid_cat = train_test_split(
        X_catboost, y, test_size=0.2, stratify=y, random_state=seed
    )
    catboost = build_catboost_model(seed)
    catboost.fit(X_train_cat, y_train_cat, cat_features=CATBOOST_CATEGORICAL_COLUMNS)
    cat_score = accuracy_score(y_valid_cat, catboost.predict(X_valid_cat))

    holdout_rows.append({
        "seed": seed,
        "logistic_holdout": float(logistic_score),
        "catboost_holdout": float(cat_score),
        "delta": float(cat_score - logistic_score),
    })

holdout_summary = pd.DataFrame(holdout_rows).sort_values("seed")
display(holdout_summary)
print('holdout delta mean:', round(float(holdout_summary['delta'].mean()), 5))


## Review memo

- seed ごとに `delta` の符号が揺れるかどうかを見る
- `delta` が小さく、CV std より十分に小さいなら採用差としては弱い
- holdout でも優位が再現しないなら、`exp013` の CV 改善はノイズ寄りと判断する
